In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# LIbraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import  RandomForestClassifier


import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Task 1: Write your code here:
df_delivery = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(df_delivery)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=['Order_ID'])
df.head()

In [ ]:
# Task 2: Write your code here:
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
missing_values = df.isnull().sum()
print(missing_values)

In [ ]:

print(f"Before: {df.shape}")
df = df.dropna(subset=['Delivery_Time','Courier_Experience_yrs'])
print(f"After dropping: {df.shape}")

for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df[col] = df[col].fillna('unknown')

df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mode())
print("Missing values remaining:", df.isnull().sum().sum())

In [ ]:
#Task 3: Write your code here:
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:

categorical_cols = ['Weather', 'Time_of_Day', 'Vehicle_Type','Courier_Experience_yrs',"Traffic_Level"]
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

df.head()

In [ ]:
# Task 5: Write your code here:
colum = ['Distance_km','Weather','Traffic_Level','Time_of_Day','Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs','Delivery_Time']
sc = StandardScaler()
data_scaled = sc.fit_transform(df[colum])

data_scaled

In [ ]:
df = pd.DataFrame(data_scaled, columns=colum)
df

In [ ]:
# Task 6: Write your code here:
df['Delivery_Time'].hist()


In [ ]:
def check_target_imbalance(df, target_column):
    print("Target Distribution:")
    print(df[target_column].value_counts(normalize=True))
    sns.countplot(x=df[target_column])
    plt.title("Target Distribution")
    plt.show()

check_target_imbalance(df, "Delivery_Time")

In [ ]:
# TARGET IS NOT BALANCED


In [ ]:
# Task 1: Write your code here:

X = df.drop("Delivery_Time", axis=1)
y = df["Delivery_Time"]

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:
rf= RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
mae = []

skf = KFold(n_splits=5)
for train_index, test_index in skf.split(X, y):
    # Split data into training and testing sets
    X_Train, X_Test = X.loc[train_index, :], X.loc[test_index, :]
    y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
    # Train the model
    rf.fit(X_Train, y_train)
    # Predict on the test set
    y_pred = rf.predict(X_Test)

    mae.append(mean_absolute_error(y_pred, y_Test))

print(f"MAE: {np.mean(mae):.4f}")


In [ ]:
# Task 1: Write your code here:
feats = ['Distance_km','Weather','Traffic_Level','Time_of_Day','Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs']

feature_importance = pd.DataFrame({
    'feature': feats,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=20, edgecolor='black')
plt.title('Predicted Delivery Time')
plt.xlabel('Prediction')
plt.ylabel('Frequency')
plt.show()

In [ ]:
pip install catboost

In [ ]:
# Task Bonus: Write your code here:
from lightgbm import LGBMClassifier
from xgboost import  XGBClassifier
from catboost import CatBoostRegressor
from sklearn.ensemble import VotingRegressor
models = {
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1),
    "CatBoost Regressor": CatBoostRegressor(verbose=0)
}

for model_name, model in models.items():
    mae = []
    # Stratified 5-Fold Cross-Validation
    skf = KFold(n_splits=5)
    for train_index, test_index in skf.split(X, y):
        # Split data into training and testing sets
        X_Train, X_Test = X.loc[train_index, :], X.loc[test_index, :]
        y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
        # Train the model
        model.fit(X_Train, y_Train)
        # Predict on the test set
        y_pred = model.predict(X_Test)

        # Calculate metrics
        mae.append(mean_absolute_error(y_Test, y_pred))

    # Print the results
    print(f"{model_name} MAE: {np.mean(mae):.4f}")
    print("\n")